# Day 05 · Senses: Sight — The Signal You Must See

**Question.** At how many of Chicago's crossings must you see the signal to know when to walk?

A walk signal is information the city gives only to the eye. An accessible pedestrian signal (APS)
also says it out loud and with a vibrating arrow, so a person who is blind or has low vision knows when
the walk phase starts. In 2023 a federal court found that Chicago's lack of them violates the ADA; its
2025 order requires APS at about 71% of signalized intersections by 2035 and at all of them by 2040,
first within 1/8 mile (one block, 100 address numbers) of transit.

**Approach.** Place every APS on the City's lists (CDOT's June 2026 report, with install dates, and the
newer ChicagoAPS.org list) on OpenStreetMap's signalized intersections. Measure, on Day 03's walk
network, which intersections lie within one block of the transit the order names, which 'L' stations
have an APS crossing within one block, and how far residents with a vision difficulty (ACS) must walk to
the nearest one.

**Output.** `output/plate.json`, read by the website to draw the plate.

| Section | What happens |
| --- | --- |
| 1. Setup | Imports, paths, run settings, figures from the documents |
| 2. Pull | APS lists from the PDF report and the City site; stations; OSM signals, streets, rail; transit feeds |
| 3. Validate | Checks that stop the run on surprises |
| 4. Transform | Signalized intersections; APS placed on them; stations; transit priority; walk distances |
| 5. Analyze | Counts, the finding |
| 6. QA | Placement against community areas, counts against CDOT, hand placements |
| 7. QA view | A quick map and chart |
| 8. Export | `plate.json` with the data ledger |

Run top to bottom from this folder. To refresh the station snapshot and transit feeds, set
`REFRESH = True` in section 1; the APS documents are dated files in `data/raw/`.

## 1. Setup

In [ ]:
from __future__ import annotations

import html
import json
import logging
import re
import subprocess
import zipfile
from datetime import date
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import shapely
from pypdf import PdfReader
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial import cKDTree

from atlas import Ledger, gtfs, snapshot, to_address, walk, write_plate

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
log = logging.getLogger('day05')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

In [ ]:
# Run settings ---------------------------------------------------------------------------------
REFRESH = False  # True = pull a new station snapshot from the Chicago Data Portal

DAY = 5
STATIONS_ID = '8pix-ypme'  # CTA - System Information - List of 'L' Stops
PORTAL = 'https://data.cityofchicago.org'
ORDER_URL = 'https://dralegal.org/wp-content/uploads/2019/09/365_Remedial_Plan_Order_ACC.pdf'
REPORT_URL = ('https://www.chicagoaps.org/wp-content/uploads/2026/09/'
              'Chicago_APS_Semi-Annual_Report_June_2026_With_Appendices2.pdf')
MONITOR_URL = 'https://dralegal.org/wp-content/uploads/2019/09/01_Plan_Year_1_Independent_Monitor_Annual_Report.pdf'
SITE_URL = 'https://www.chicagoaps.org/existing-aps-locations/'

# Paths: the notebook runs from its own folder (days/day05-sight/)
DAY_DIR = Path.cwd()
REPO = DAY_DIR.parents[1]
RAW_DIR = DAY_DIR / 'data' / 'raw'
PROC_DIR = DAY_DIR / 'data' / 'processed'
OUT_PATH = DAY_DIR / 'output' / 'plate.json'
OSM_DIR = REPO / walk.OSM_DIR
BLOCKS_CSV = REPO / 'reference' / 'chicago-blocks-2020.csv'
BOUNDARY_RAW = sorted((REPO / 'reference' / 'raw').glob('city_boundary_*.geojson'))[-1]
GTFS_ZIP = sorted((REPO / 'reference' / 'raw' / 'gtfs').glob('cta_gtfs_*.zip'))[-1]
REPORT_PDF = RAW_DIR / 'cdot_aps_semiannual_2026-06.pdf'
SITE_HTML = sorted(RAW_DIR.glob('chicagoaps_existing_*.html'))[-1]
assert (DAY_DIR / 'day05.ipynb').exists(), f'Run this notebook from its own folder, not {DAY_DIR}'

CRS = walk.CRS
LINES = ('Red', 'Blue', 'Brown', 'Green', 'Orange', 'Purple', 'Pink', 'Yellow')
DAY01_COUNTS = {'total': 144, 'noStepFree': 38}  # Plate I, from the same station dataset

# From the documents (read 2026-11-05; page numbers in the ledger) ------------------------------
CDOT = {'aps': 165, 'signals': 2722, 'asOf': '2026-06-12', 'year1': 78}   # Semi-Annual Report, June 2026, p. 2
ORDER = {'entered': '2025-05-29', 'phase1': 0.71, 'phase1By': 2035, 'allBy': 2040,
         'minimums': [70, 110, 135, 154, 185, 220, 235, 245, 245, 245]}  # Dkt. 365, pp. 5-7
MONITOR = {'inspected': 103, 'compliant': 0}                               # Plan Year 1 report
ONE_BLOCK_M = 1609.344 / 8   # the order's 1/8 mile: one block, 100 address numbers
SIGNAL_SPAN_M = 80           # signal and crosswalk nodes of one intersection lie within this of each other
APS_SNAP_M = 60              # a geocoded APS must lie this close to a signalized intersection

## 2. Pull

**APS lists.** CDOT's June 2026 Semi-Annual Report lists every intersection with APS and its install
date (Appendix A, 165 rows as of 12 June 2026). ChicagoAPS.org, the City's APS site, has the newer
list (no dates). Both are saved in `data/raw/` with the court order and the monitor's report.

**Signals.** There is no City dataset of traffic signals. OpenStreetMap maps each signal head
(`highway=traffic_signals`) and each signalized crosswalk (`crossing=traffic_signals`); they are
merged into intersections below and checked against CDOT's count of 2,722.

In [ ]:
DATE_AT_END = re.compile(r'^(?P<loc>.*?\S)\s*(?P<date>\d{1,2}/\d{1,2}/\d{4})\s*$')

report_rows = []
for page_no, page in enumerate(PdfReader(REPORT_PDF).pages, start=1):
    text = page.extract_text() or ''
    if 'Intersections with APS Installed' not in text:
        continue
    for line in text.splitlines():
        if (m := DATE_AT_END.match(line.strip())) and line.strip() != CDOT['asOf']:
            report_rows.append({'location': m['loc'].strip(), 'installed': pd.to_datetime(m['date']), 'page': page_no})
    if (m := re.search(r'Total:\s*(\d+)', text)):
        report_total = int(m[1])
report = pd.DataFrame(report_rows)
report = report[~report['location'].str.fullmatch(r'\d{1,2}/\d{1,2}/\d{4}')].reset_index(drop=True)

cell = lambda s: html.unescape(re.sub(r'<[^>]+>', '', s)).strip()
site_html = SITE_HTML.read_text()
site = pd.DataFrame([(cell(a), cell(b)) for a, b in
                     re.findall(r'<td[^>]*>(.*?)</td>\s*<td[^>]*>(.*?)</td>', site_html, re.S)],
                    columns=['location', 'community'])
site_as_of = pd.to_datetime(re.search(r'as of\s*(\d{1,2}/\d{1,2}/\d{4})', cell(site_html))[1]).date()
site_pulled = date.fromisoformat(SITE_HTML.stem.rsplit('_', 1)[-1])
log.info('Report: %d rows (total line %d); site: %d rows, %d distinct, as of %s',
         len(report), report_total, len(site), site['location'].nunique(), site_as_of)

stations_raw, stations_pulled = snapshot(STATIONS_ID, RAW_DIR, 'l_stops', refresh=REFRESH,
                                         select='map_id,station_descriptive_name,ada,location')
blocks = pd.read_csv(BLOCKS_CSV, dtype={'geoid': str})
city = shapely.union_all(gpd.read_file(BOUNDARY_RAW).to_crs(CRS).geometry)

In [ ]:
# OSM: signal nodes, named streets and the elevated rail, cut from the newest Illinois extract.
pbf, osm_date = walk.latest_pbf(OSM_DIR)
PROC_DIR.mkdir(parents=True, exist_ok=True)
box = PROC_DIR / f'chicago-{osm_date.isoformat()}.osm.pbf'
layers = {
    'signals': ['n/highway=traffic_signals', 'n/crossing=traffic_signals'],
    'streets': ['w/highway=motorway,trunk,primary,secondary,tertiary,unclassified,residential,living_street,'
                'motorway_link,trunk_link,primary_link,secondary_link,tertiary_link,service,pedestrian'],
    'rail': ['w/railway=subway,light_rail,rail'],
}
if not box.exists():
    subprocess.run(['osmium', 'extract', '-b', '-87.95,41.64,-87.52,42.03', str(pbf), '-o', str(box), '--overwrite'], check=True)
for name, filt in layers.items():
    out = PROC_DIR / f'chicago-{name}-{osm_date.isoformat()}.osm.pbf'
    if not out.exists():
        omit = ['-R'] if name == 'signals' else []  # ways need their nodes
        subprocess.run(['osmium', 'tags-filter', *omit, str(box), *filt, '-o', str(out), '--overwrite'], check=True)

sig_nodes = gpd.read_file(PROC_DIR / f'chicago-signals-{osm_date.isoformat()}.osm.pbf', layer='points',
                          columns=['osm_id', 'highway', 'other_tags']).to_crs(CRS)
streets = gpd.read_file(PROC_DIR / f'chicago-streets-{osm_date.isoformat()}.osm.pbf', layer='lines',
                        columns=['osm_id', 'name', 'highway', 'other_tags']).to_crs(CRS)
rail = gpd.read_file(PROC_DIR / f'chicago-rail-{osm_date.isoformat()}.osm.pbf', layer='lines',
                     columns=['osm_id', 'name', 'railway', 'other_tags']).to_crs(CRS)
log.info('OSM %s: %d signal nodes, %d street ways, %d rail ways', osm_date, len(sig_nodes), len(streets), len(rail))

In [ ]:
# Transit layers for the order's priority rule: CTA (Plate I and II's feed), Metra and Pace schedule
# feeds, saved dated beside CTA's in reference/raw/gtfs/ (not committed).
GTFS_DIR = REPO / 'reference' / 'raw' / 'gtfs'
FEEDS = {'metra': 'https://schedules.metrarail.com/gtfs/schedule.zip', 'pace': 'https://www.pacebus.com/gtfsdownload'}
feed_zip = {}
for name, url in FEEDS.items():
    have = sorted(GTFS_DIR.glob(f'{name}_gtfs_*.zip'))
    if REFRESH or not have:
        path = GTFS_DIR / f'{name}_gtfs_{date.today().isoformat()}.zip'
        path.write_bytes(requests.get(url, timeout=120).content)
        have = [path]
    feed_zip[name] = have[-1]
read = lambda z, f, **kw: pd.read_csv(zipfile.ZipFile(z).open(f), dtype=str, skipinitialspace=True, **kw)
log.info('Feeds: %s; %s', GTFS_ZIP.name, ', '.join(p.name for p in feed_zip.values()))

## 3. Validate

In [ ]:
tag = lambda key: sig_nodes['other_tags'].fillna('').str.extract(rf'"{key}"=>"([^"]*)"')[0]
sig_nodes['crossing'] = tag('crossing')
sig_nodes = sig_nodes[sig_nodes.within(city)].reset_index(drop=True)
EXPECTED_ST = {'map_id', 'station_descriptive_name', 'ada', 'location.latitude', 'location.longitude'}
checks = {
    'report rows = its total line': len(report) == report_total == CDOT['aps'],
    'report dates parse': report['installed'].notna().all(),
    'report dates up to the as-of date': (report['installed'] <= pd.Timestamp(CDOT['asOf'])).all(),
    'site list as of 2026-11-01 or later': site_as_of >= date(2026, 11, 1),
    'site rows have a community area': site['community'].ne('').all(),
    'station columns': EXPECTED_ST <= set(stations_raw.columns),
    'blocks add up to 2020 Chicago': blocks['pop'].sum() == 2_746_388,
    'blocks carry vision difficulty': 'vis' in blocks.columns,
    'OSM signals in the city': len(sig_nodes) > 10_000,
}
for name, ok in checks.items():
    log.info('%-40s %s', name, 'ok' if ok else 'FAILED')
assert all(checks.values()), 'Validation failed; see log above'

## 4. Transform

**Signalized intersections.** A typical OSM intersection has one signal node and four signalized
crosswalk nodes. Nodes are grouped by complete linkage, so every group spans at most 80 m: large
divided intersections stay whole while the next intersection, a block (100 m or more) away, stays
apart. (Single linkage chained neighbours together along dense streets such as Ashland.) An
intersection counts as having **pedestrian signals** when one of its crosswalks is tagged
`crossing=traffic_signals`, the universe the court order uses, or when it has an APS (section 4,
below). Its position is the mean of its nodes.

In [ ]:
XY = sig_nodes.get_coordinates().to_numpy()
sig_nodes['cluster'] = fcluster(linkage(XY, 'complete'), SIGNAL_SPAN_M, 'distance')
ped = sig_nodes['crossing'].eq('traffic_signals')
inter = (sig_nodes.assign(x=XY[:, 0], y=XY[:, 1], ped=ped, head=sig_nodes['highway'].eq('traffic_signals'))
         .groupby('cluster').agg(x=('x', 'mean'), y=('y', 'mean'), nodes=('x', 'size'),
                                 ped=('ped', 'any'), head=('head', 'any')))
log.info('Signal clusters: %d; with pedestrian signals: %d (CDOT: %d); heads without a signalized crosswalk: %d',
         len(inter), inter['ped'].sum(), CDOT['signals'], (~inter['ped']).sum())
# every cluster is kept for now: an APS at a cluster with no signalized crosswalk tag adds it
signals = inter.reset_index().rename(columns={'cluster': 'osm_cluster'})
signals['sid'] = np.arange(len(signals))
SIG_XY = signals[['x', 'y']].to_numpy()
sig_tree = cKDTree(SIG_XY)

**Placing the APS list.** Each listed location names two or more streets ("N PULASKI RD-N ELSTON
AVE", "Ashland Ave & Devon Ave & Clark St") or an address on one street ("2420 S HALSTED ST", a
mid-block crossing). Street names are matched to OSM names with the direction dropped and words
shortened as Day 04 ("Doctor Martin Luther King Junior Drive" = "DR MARTIN LUTHER KING JR DR"); where
the suffix differs ("Diversey Ave" for Diversey Parkway) the bare name is tried. Congress Parkway
and Ida B. Wells Drive are one street.

- An **intersection** is where the named streets meet (within 40 m, so T junctions and divided
  streets count), assigned to the signalized intersection nearest the most meeting points, within
  60 m (150 m at expressway ramps, whose signals sit off the bridge).
- An **address** is placed on its street at that number on the grid (800 to the mile) and assigned
  to the nearest signal on the same street within 150 m.
- Where the streets meet but OSM maps **no signal**, the APS itself shows that a pedestrian signal
  is there: the location is added to the signalized intersections and counted (QA, section 6).

A few locations are rewritten by hand first, each with its reason.

In [ ]:
REWRITE = {  # listed text -> text the matcher can read
    'N SHERIDAN RD-N SHERIDAN RD@6420N (N. INT.)': '6420 N SHERIDAN RD',  # the address is in the name
    '51050 W 55th St': '5150 W 55th St',        # site typo for the report's 5150 W 55TH ST DRIVEWAY
    'Milwaukee Ave & Logan South Drive': 'Milwaukee Ave & Logan Blvd',  # Logan Blvd's south drive at the square
    # streets OSM does not name or join: placed by the grid address of the cross street
    'N PULASKI RD-W THORNDALE AVE (PVT)': '5900 N PULASKI RD',  # Thorndale is 5900 N; the private part is unnamed
    'Pulaski Rd & Thorndale Ave (Pvt)': '5900 N PULASKI RD',
    'W LAKE ST & N RIVERSIDE PLAZA': '400 W LAKE ST',           # Riverside Plaza runs along the river at 400 W
    'Lake St & Riverside Plaza': '400 W LAKE ST',
    '858 W Harrison Crossing': '848 W HARRISON CROSSING',      # the site's number for the report's one crossing
    # Edgebrook Golf Course fronts Central Ave at about 5900-6400 N (OSM), not 3000 N: left out
    '3000 N CENTRAL EDGEBROOK GOLF COURSE DRIVEWAY': None,
    '3000 N Central Edgebrook Golf Course': None,
}
SUFFIX = {'AVENUE': 'AVE', 'STREET': 'ST', 'ROAD': 'RD', 'BOULEVARD': 'BLVD', 'DRIVE': 'DR', 'COURT': 'CT',
          'PLACE': 'PL', 'PARKWAY': 'PKWY', 'HIGHWAY': 'HWY', 'TERRACE': 'TER', 'EXPRESSWAY': 'EXPY',
          'PLAZA': 'PLZ', 'LANE': 'LN', 'SQUARE': 'SQ'}
DIRS = {'NORTH', 'SOUTH', 'EAST', 'WEST', 'N', 'S', 'E', 'W'}
ALIAS = {'DR MARTIN LUTHER KING JR DR': 'MARTIN LUTHER KING JR DR',
         'JEAN BAPTISTE POINT DUSABLE LAKE SHORE DR': 'LAKE SHORE DR', 'DUSABLE LAKE SHORE DR': 'LAKE SHORE DR',
         'LAKE SHORE DRE': 'LAKE SHORE DR',  # typo in the report
         'BROADWAY AVE': 'BROADWAY', 'BROADWAY ST': 'BROADWAY', 'LA SALLE ST': 'LASALLE ST',
         'LA SALLE DR': 'LASALLE DR', 'CONGRESS PKWY': 'IDA B WELLS DR', 'EDENS RAMP': 'EDENS EXPY'}
ADDRESS = re.compile(r'^(?P<num>\d+)\s+(?P<dir>[NSEW])\.?\s+(?P<rest>.+)$', re.I)


def bare(n: str) -> str:
    """The name without its suffix: 'DIVERSEY PKWY' -> 'DIVERSEY'."""
    w = n.split()
    return ' '.join(w[:-1]) if len(w) > 1 and w[-1] in set(SUFFIX.values()) else n


WORDS = {'DOCTOR': 'DR', 'JUNIOR': 'JR', 'SAINT': 'ST'}
LEVELS = {'UPPER', 'LOWER'}
MEET_M, RAMP_SNAP_M, ADDRESS_SNAP_M, ON_STREET_M = 40, 150, 150, 25


def norm(name) -> str:
    """'West North Avenue' -> 'NORTH AVE', 'Lower Wacker Drive' -> 'WACKER DR', 'S AVENUE O' -> 'AVE O'."""
    if not isinstance(name, str):
        return ''
    name = re.sub(r'\(.*?\)', ' ', name.upper())
    words = [WORDS.get(w, w) for w in re.sub(r'[^A-Z0-9 ]', ' ', name).split()]
    words = [w for i, w in enumerate(words) if not (i == 0 and w in LEVELS)]
    if len(words) > 1 and words[0] in DIRS and not (len(words) == 2 and words[1] in SUFFIX.values() | SUFFIX.keys()):
        words = words[1:]
    words = [w for i, w in enumerate(words) if not (i == 0 and w in LEVELS)]
    s = ' '.join(SUFFIX.get(w, w) for w in words)
    return ALIAS.get(s, s)


streets['key'] = streets['name'].map(norm)
streets['bare'] = streets['key'].map(bare)
by_key = {k: shapely.union_all(g.geometry.values) for k, g in streets[streets['key'] != ''].groupby('key')}
by_bare = {k: shapely.union_all(g.geometry.values) for k, g in streets[streets['bare'] != ''].groupby('bare')}


def street(part: str):
    """The named street's geometry; a name that starts like a direction ('North Shore Ave') is tried
    whole too."""
    for k in (norm('W ' + part), norm(part)):  # 'East River Rd' is East River Road, not River Road
        if (g := by_key.get(k)) is not None:
            return g
    return by_bare.get(bare(norm(part)))


def meeting_points(a, b) -> np.ndarray:
    """Where two streets cross, or, if they only come close (a T, a divided street), the midpoint of
    each close approach."""
    x = shapely.get_coordinates(shapely.intersection(a, b))
    if len(x):
        return x
    pts = []
    for part in shapely.get_parts(a):
        if shapely.distance(part, b) <= MEET_M:
            line = shapely.shortest_line(part, b)
            pts.append(shapely.get_coordinates(line).mean(axis=0))
    return np.array(pts).reshape(-1, 2)


def on_street(geom, k: np.ndarray) -> np.ndarray:
    return shapely.distance(shapely.points(SIG_XY[k]), geom) <= ON_STREET_M


def place(location: str) -> tuple[np.ndarray | None, int, str]:
    """A listed APS location -> (x, y) in meters, signal id (-1 = none in OSM) and how it was placed."""
    loc = REWRITE.get(location.strip(), location.strip())
    if loc is None:
        return None, -1, 'left out (see REWRITE)'
    if (m := ADDRESS.match(loc)):
        num, d, rest = int(m['num']), m['dir'].upper(), m['rest']
        words = re.sub(r'\(.*?\)', ' ', rest).upper().split()
        geom = next((g for n in range(len(words), 0, -1) if (g := street(' '.join(words[:n]))) is not None), None)
        if geom is None:
            return None, -1, 'street not found'
        coords = shapely.get_coordinates(shapely.segmentize(geom, 5))
        ll = gpd.GeoSeries(gpd.points_from_xy(*coords.T), crs=CRS).to_crs('EPSG:4326')
        ax, ay = to_address(ll.y.to_numpy(), ll.x.to_numpy())
        along = ay if d in 'NS' else ax
        i = int(np.argmin(np.abs(along - (-num if d in 'SW' else num))))
        if abs(along[i] - (-num if d in 'SW' else num)) > 50:
            return None, -1, 'address beyond the street'
        xy = coords[i]
        near = np.array(sig_tree.query_ball_point(xy, ADDRESS_SNAP_M), dtype=int)
        near = near[on_street(geom, near)] if len(near) else near
        if len(near):
            k = near[np.argmin(np.hypot(*(SIG_XY[near] - xy).T))]
            return SIG_XY[k], int(k), 'address'
        return xy, -1, 'address, no signal in OSM'

    qualifier = re.search(r'\b(NORTH|SOUTH)\)?\s*$', loc.upper())
    loc = re.sub(r'\s+\(?(NORTH|SOUTH)\)?\s*$', '', loc, flags=re.I)
    parts = [p for p in re.split(r'\s+-\s+|-|\s*&\s*', loc) if p.strip()]
    parts = list({norm(p): p for p in parts}.values())  # 'N ASHLAND AVE-S ASHLAND AVE-...' names one street
    geoms = [street(p) for p in parts]
    found = [g for g in geoms if g is not None]
    if len(found) < 2:
        return None, -1, f'not found: {[p for p, g in zip(parts, geoms) if g is None]}'
    pairs = lambda gs: np.concatenate([meeting_points(gs[i], gs[j]) for i in range(len(gs)) for j in range(i + 1, len(gs))
                                       if gs[i] is not gs[j]] or [np.empty((0, 2))])
    hits = pairs(found)
    if not len(hits):  # 'Diversey Ave' for Diversey Parkway, '57th St' for 57th Drive: try the bare names
        loose = [by_bare.get(bare(norm(p))) for p in parts]
        loose = [g for g in loose if g is not None]
        hits = pairs(loose) if len(loose) >= 2 else hits
    if not len(hits):
        return None, -1, 'streets do not meet'
    snap = RAMP_SNAP_M if 'RAMP' in loc.upper() else APS_SNAP_M
    d, k = sig_tree.query(hits, distance_upper_bound=snap)
    ok = np.isfinite(d)
    if not ok.any():
        # one meeting point stands for the location: the one nearest the others
        mid = hits[np.argmin([np.hypot(*(hits - h).T).sum() for h in hits])]
        return mid, -1, 'intersection, no signal in OSM'
    d, k = d[ok], k[ok]
    if qualifier:  # Kedzie & Milwaukee North / South: the two crossings at Logan Square
        cand = np.unique(k)
        pick = cand[np.argmax(SIG_XY[cand, 1])] if qualifier[1] == 'NORTH' else cand[np.argmin(SIG_XY[cand, 1])]
    else:
        votes = pd.Series(k).value_counts()
        top = votes[votes == votes.max()].index
        pick = min(top, key=lambda s: d[k == s].min())
    return SIG_XY[pick], int(pick), 'intersection'


for df in (report, site):
    placed = [place(s) for s in df['location']]
    df['x'] = [p[0][0] if p[0] is not None else np.nan for p in placed]
    df['y'] = [p[0][1] if p[0] is not None else np.nan for p in placed]
    df['sid'] = [p[1] for p in placed]
    df['how'] = [p[2] for p in placed]
for df in (report, site):
    log.info('Placed: %s', df['how'].value_counts().to_dict())

**One list.** Locations OSM has no signal for become new signalized intersections (one per place:
the report and the site name the same corner in different words). The report gives install dates;
a location only on the newer site list was installed between 12 June and 1 November 2026.

In [ ]:
new = pd.concat([report, site])
new = new[(new['sid'] < 0) & new['x'].notna()].drop_duplicates(['x', 'y'])
if len(new):
    lab = fcluster(linkage(new[['x', 'y']].to_numpy(), 'complete'), SIGNAL_SPAN_M, 'distance') if len(new) > 1 else [1]
    add = new.assign(c=lab).groupby('c').agg(x=('x', 'mean'), y=('y', 'mean'), location=('location', 'first'))
    add = add.assign(osm_cluster=-1, nodes=0, ped=True, head=False, added=True, sid=len(signals) + np.arange(len(add)))
    signals = pd.concat([signals.assign(added=False), add.drop(columns='location')], ignore_index=True)
    SIG_XY = signals[['x', 'y']].to_numpy()
    sig_tree = cKDTree(SIG_XY)
    for df in (report, site):
        lost = (df['sid'] < 0) & df['x'].notna()
        df.loc[lost, 'sid'] = sig_tree.query(df.loc[lost, ['x', 'y']].to_numpy())[1]
log.info('Added %d signalized intersections that OSM lacks', int(signals['added'].sum()))

dup_report = report[report['sid'] >= 0].duplicated('sid', keep=False)
dup_site = site[site['sid'] >= 0].drop_duplicates('location').duplicated('sid', keep=False)
print('report names sharing a signal:', report.loc[dup_report[dup_report].index, ['location', 'sid']].to_string())
print('site names sharing a signal:', site.loc[dup_site[dup_site].index, ['location', 'sid']].to_string())

placed_r = report[report['sid'] >= 0]
placed_s = site[site['sid'] >= 0]
aps = (pd.DataFrame({'sid': sorted(set(placed_r['sid']) | set(placed_s['sid']))})
       .merge(placed_r.groupby('sid').agg(installed=('installed', 'min'), name=('location', 'first')).reset_index(), how='left')
       .merge(placed_s.groupby('sid').agg(site_name=('location', 'first'), community=('community', 'first')).reset_index(), how='left'))
aps['onReport'] = aps['installed'].notna()
aps['onSite'] = aps['site_name'].notna()
print(aps[['onReport', 'onSite']].value_counts().to_string())
print('report only:', aps.loc[~aps['onSite'], 'name'].tolist())
print('site only:', aps.loc[~aps['onReport'], 'site_name'].tolist())

**Stations.** Day 01's rule, unchanged: a station is vertically accessible only if every one of its
stops is flagged accessible. Stations outside the city (Evanston, Oak Park and the rest) are kept
for the chart but left out of the counts: APS are a City program.

In [ ]:
PAREN = re.compile(r'\((?P<inside>[^)]*)\)\s*$')
stops = stations_raw[list(EXPECTED_ST)].copy()
stops['ada'] = stops['ada'].astype(str).str.lower().str.strip()
stops['lat'] = pd.to_numeric(stops['location.latitude'], errors='coerce')
stops['lon'] = pd.to_numeric(stops['location.longitude'], errors='coerce')
stations = (stops.assign(step_free=stops['ada'].eq('true'))
            .groupby('map_id', as_index=False)
            .agg(name=('station_descriptive_name', 'first'), step_free=('step_free', 'all'),
                 lat=('lat', 'mean'), lon=('lon', 'mean')))
stations['name'] = stations['name'].str.replace(r'\s+', ' ', regex=True).str.strip()
stations['short'] = stations['name'].map(lambda s: PAREN.sub('', s).strip())


def parse_lines(name: str) -> list[str]:
    """The 'L' lines named in the station's parenthetical, in canonical order."""
    m = PAREN.search(name)
    return [l for l in LINES if re.search(rf'\b{l}\b', m['inside'] if m else '')]


stations['lines'] = stations['name'].map(parse_lines)
counts = {'total': len(stations), 'noStepFree': int((~stations['step_free']).sum())}
assert counts == DAY01_COUNTS, f'Station counts changed since Plate I: {counts}'
st_m = gpd.GeoSeries(gpd.points_from_xy(stations['lon'], stations['lat']), crs='EPSG:4326').to_crs(CRS)
ST_XY = st_m.get_coordinates().to_numpy()
stations['inCity'] = st_m.within(city).to_numpy()
log.info('Stations: %d, %d inside the city', len(stations), stations['inCity'].sum())

**The 'L' track.** The rail shapes in CTA's schedule feed, as Plate II: every shape that carries at
least a tenth of its line's trips (both Green Line branches, the Purple express to the Loop). Drawn
for context only.

In [ ]:
RAIL_NAMES = {'Red': 'Red', 'Blue': 'Blue', 'Brn': 'Brown', 'G': 'Green', 'Org': 'Orange', 'P': 'Purple', 'Pink': 'Pink', 'Y': 'Yellow'}
rail_routes = read(GTFS_ZIP, 'routes.txt').query('route_type == "1"')
rail_trips = read(GTFS_ZIP, 'trips.txt', usecols=['route_id', 'shape_id'])
rail_trips = rail_trips[rail_trips['route_id'].isin(rail_routes['route_id'])]
rail_counts = rail_trips.groupby(['route_id', 'shape_id']).size()
rail_counts = rail_counts[rail_counts >= 0.1 * rail_counts.groupby(level=0).transform('sum')]
rail_paths = gtfs.shape_paths(GTFS_ZIP, set(rail_counts.index.get_level_values(1)))
track = [{'line': RAIL_NAMES[rid],
          'path': [[round(x), round(y)] for x, y in shapely.simplify(shapely.LineString(rail_paths[sid][['x', 'y']].to_numpy()), 10).coords]}
         for rid, sid in rail_counts.index]
assert {t['line'] for t in track} == set(LINES), 'A line has no rail shape'
log.info('Rail shapes: %d on %d lines, %d vertices', len(track), len(LINES), sum(len(t['path']) for t in track))

**The order's transit priority** (§ III.A.2.d): signalized intersections within 1/8 mile of CTA
stations, Metra stations, Pace stations, bus-route-to-bus-route transfer points, and streets that
pass under elevated train lines. Each is turned into points:

- CTA: the 'L' stations above. Metra: the stations in Metra's feed. Pace: the stops of its Pulse
  rapid-bus lines (Pace's stations).
- Transfer points: where two CTA bus routes cross (route shapes from the feed; shared stretches of
  street are not crossings).
- Under the L: where a street passes under an elevated CTA track (OSM ways tagged as a bridge or
  above ground level).

In [ ]:
def to_m(lat, lon) -> np.ndarray:
    return gpd.GeoSeries(gpd.points_from_xy(pd.to_numeric(lon), pd.to_numeric(lat)), crs='EPSG:4326').to_crs(CRS).get_coordinates().to_numpy()


def near_city(xy: np.ndarray, margin: float = ONE_BLOCK_M) -> np.ndarray:
    """Points in the city or within a block of it (a station just outside still counts)."""
    return shapely.contains_xy(city.buffer(margin), xy[:, 0], xy[:, 1])


metra = read(feed_zip['metra'], 'stops.txt')
metra_xy = to_m(metra['stop_lat'], metra['stop_lon'])
metra_xy = metra_xy[near_city(metra_xy)]

pace_routes = read(feed_zip['pace'], 'routes.txt')
pulse = set(pace_routes.loc[pace_routes['route_long_name'].str.contains('Pulse', na=False), 'route_id'])
pace_trips = read(feed_zip['pace'], 'trips.txt')
pulse_trips = set(pace_trips.loc[pace_trips['route_id'].isin(pulse), 'trip_id'])
pulse_stops = set()
for chunk in read(feed_zip['pace'], 'stop_times.txt', usecols=['trip_id', 'stop_id'], chunksize=1_000_000):
    pulse_stops |= set(chunk.loc[chunk['trip_id'].isin(pulse_trips), 'stop_id'])
pace_stops = read(feed_zip['pace'], 'stops.txt')
pace_stops = pace_stops[pace_stops['stop_id'].isin(pulse_stops)]
pace_xy = to_m(pace_stops['stop_lat'], pace_stops['stop_lon'])
pace_xy = pace_xy[near_city(pace_xy)]

cta_routes = read(GTFS_ZIP, 'routes.txt')
bus_routes = set(cta_routes.loc[cta_routes['route_type'] == '3', 'route_id'])
cta_trips = read(GTFS_ZIP, 'trips.txt', usecols=['route_id', 'shape_id'])
route_shapes = cta_trips[cta_trips['route_id'].isin(bus_routes)].drop_duplicates()
shp = read(GTFS_ZIP, 'shapes.txt')
shp = shp[shp['shape_id'].isin(set(route_shapes['shape_id']))]
shp['seq'] = shp['shape_pt_sequence'].astype(int)
shp_xy = to_m(shp['shape_pt_lat'], shp['shape_pt_lon'])
shp['x'], shp['y'] = shp_xy[:, 0], shp_xy[:, 1]
lines_by_shape = {sid: shapely.LineString(g.sort_values('seq')[['x', 'y']].to_numpy()) for sid, g in shp.groupby('shape_id')}
route_geom = {r: shapely.simplify(shapely.union_all([lines_by_shape[s] for s in g['shape_id'] if s in lines_by_shape]), 5)
              for r, g in route_shapes.groupby('route_id')}


def heading(geom, xy: np.ndarray, step: float = 20) -> np.ndarray:
    """Direction (radians, 0..pi) of a route at each point, from the nearest of its pieces."""
    parts = shapely.get_parts(shapely.line_merge(geom) if geom.geom_type == 'MultiLineString' else geom)
    tree = shapely.STRtree(parts)
    pts = shapely.points(xy)
    line = parts[tree.query_nearest(pts, all_matches=False)[1]]
    at = shapely.line_locate_point(line, pts)
    a, b = shapely.line_interpolate_point(line, at - step), shapely.line_interpolate_point(line, at + step)
    d = shapely.get_coordinates(b) - shapely.get_coordinates(a)
    return np.arctan2(d[:, 1], d[:, 0]) % np.pi


rids = sorted(route_geom)
cross = []
for i, a in enumerate(rids):
    for b in rids[i + 1:]:
        x = shapely.intersection(route_geom[a], route_geom[b])
        pts = shapely.get_coordinates([p for p in shapely.get_parts(x) if p.geom_type == 'Point'])
        if len(pts):
            ang = np.abs(heading(route_geom[a], pts) - heading(route_geom[b], pts))
            ang = np.minimum(ang, np.pi - ang)
            cross.extend(pts[ang > np.radians(30)])  # routes that cross, not two that share a street
cross = np.array(cross)
cross = cross[near_city(cross, 0)]
lab = fcluster(linkage(cross, 'complete'), SIGNAL_SPAN_M, 'distance')  # one point per street corner
transfer_xy = pd.DataFrame(cross).groupby(lab).mean().to_numpy()

rail['bridge'] = rail['other_tags'].fillna('').str.contains(r'"bridge"=>"(?:yes|viaduct)"')
rail['layer'] = pd.to_numeric(rail['other_tags'].fillna('').str.extract(r'"layer"=>"(-?\d+)"')[0], errors='coerce').fillna(0)
rail['tunnel'] = rail['other_tags'].fillna('').str.contains('"tunnel"=>')
rail['cta'] = rail['railway'].isin(['subway', 'light_rail']) | rail['other_tags'].fillna('').str.contains('Chicago Transit Authority|CTA')
elevated = rail[rail['cta'] & (rail['bridge'] | (rail['layer'] > 0)) & ~rail['tunnel']]
el_geom = shapely.union_all(elevated.geometry.values)
road = streets[~streets['highway'].isin(['motorway', 'service', 'pedestrian'])]
under = shapely.get_coordinates(shapely.intersection(el_geom, shapely.union_all(road.geometry.values)))
under = under[near_city(under, 0)]
lab = fcluster(linkage(under, 'complete'), SIGNAL_SPAN_M, 'distance')  # tracks and carriageways: one point
under_xy = pd.DataFrame(under).groupby(lab).mean().to_numpy()

priority_pts = {'cta': ST_XY[stations['inCity'].to_numpy() | near_city(ST_XY)], 'metra': metra_xy, 'pace': pace_xy,
                'transfer': transfer_xy, 'underL': under_xy}
log.info('Priority points: %s; elevated CTA track %.0f km', {k: len(v) for k, v in priority_pts.items()},
         el_geom.length / 1000)

**The signal universe for the map.** OSM intersections with pedestrian signals, plus those with an
APS (an APS needs a pedestrian signal), plus the ones OSM lacks. Its count is checked against CDOT's
2,722 in section 6.

In [ ]:
signals['aps'] = signals['sid'].isin(aps['sid'])
signals['ped'] = signals['ped'] | signals['aps']
signals = signals[signals['ped']].reset_index(drop=True)
signals = signals.merge(aps[['sid', 'installed', 'name', 'site_name', 'onReport', 'onSite']], on='sid', how='left')
signals = signals[shapely.contains_xy(city.buffer(30), signals['x'], signals['y'])].reset_index(drop=True)
SIG_XY = signals[['x', 'y']].to_numpy()
N, A = len(signals), int(signals['aps'].sum())
log.info('Map universe: %d signalized intersections with pedestrian signals, %d with APS', N, A)

**Distances on the walk network** (Day 03's network: OSMnx walk rules on the same extract). One
block is the order's 1/8 mile, 201 m, measured as a walk, not a straight line. For each signal: is
it within a block of each kind of transit point? For each station and each block of residents:
how far is the walk to the nearest signal, and to the nearest APS?

In [ ]:
xml = OSM_DIR / f'chicago-walk-{osm_date.isoformat()}.osm'
g_walk = walk.main_component(walk.load(xml))
E = walk.Edges.of(g_walk)
BL_XY = to_m(blocks['lat'], blocks['lon'])
SNAP_M = 150  # as Day 03: a point farther than this from any walkway is left out


def walk_to(sources: np.ndarray, targets: np.ndarray, limit: float) -> np.ndarray:
    """Network distance from each target to the nearest source (inf beyond ``limit``)."""
    if (walk.snap(E, sources)[2] > SNAP_M).all():  # O'Hare's station is inside the terminal, off the network
        return np.full(len(targets), np.inf)
    keys = [str(i) for i in range(len(sources))]
    dist, label, _ = walk.reach(E, keys, sources, limit=limit, max_snap=SNAP_M)
    d, _ = walk.point_distance(E, dist, label, targets)
    return d


for kind, pts in priority_pts.items():
    signals[f'near_{kind}'] = walk_to(pts, SIG_XY, ONE_BLOCK_M) <= ONE_BLOCK_M
KINDS = list(priority_pts)
signals['priority'] = signals[[f'near_{k}' for k in KINDS]].any(axis=1)

APS_XY = SIG_XY[signals['aps'].to_numpy()]
stations['toSignal'] = walk_to(SIG_XY, ST_XY, 3_000)
stations['toAps'] = walk_to(APS_XY, ST_XY, 20_000)
blocks['toAps'] = walk_to(APS_XY, BL_XY, 20_000)
# signals within a block of each station (a signal can be near two stations)
near_st = []
for i, xy in enumerate(ST_XY):
    d = walk_to(xy[None, :], SIG_XY, ONE_BLOCK_M)
    near_st.append(np.flatnonzero(d <= ONE_BLOCK_M))
stations['signals'] = [len(k) for k in near_st]
stations['apsSignals'] = [int(signals['aps'].to_numpy()[k].sum()) for k in near_st]
stations['seenOnly'] = stations['signals'] - stations['apsSignals']

## 5. Analyze

In [ ]:
def share(v: float) -> str:
    """A share in words when close to a simple fraction, else a percentage (as the registry)."""
    words = [(1 / 10, 'one in ten'), (1 / 4, 'a quarter'), (1 / 3, 'one in three'), (1 / 2, 'half'),
             (2 / 3, 'two in three'), (3 / 4, 'three in four')]
    for f, w in words:
        if abs(v - f) < 0.02:
            return w if abs(v - f) < 0.007 else (f'nearly {w}' if v < f else f'just over {w}')
    return f'{100 * v:.0f}%'


city_st = stations[stations['inCity']].copy()
city_st['class'] = np.select([city_st['apsSignals'] > 0, city_st['signals'] > 0], ['aps', 'seen'], 'none')
acc = city_st[city_st['step_free']]
vis_total = blocks['vis'].sum()
vis_far = blocks.loc[blocks['toAps'] > ONE_BLOCK_M, 'vis'].sum()
order = blocks.sort_values('toAps')
vis_median_m = float(order['toAps'].to_numpy()[np.searchsorted(order['vis'].cumsum().to_numpy(), vis_total / 2)])

inst = report.drop_duplicates('location')
by_year = inst.groupby(inst['installed'].dt.year).size()
plan_year_1 = int(inst['installed'].between(pd.Timestamp(ORDER['entered']), pd.Timestamp('2026-05-28')).sum())
site_new = int((aps['onSite'] & ~aps['onReport']).sum())
listed_now = int(site['location'].nunique())  # the City's current list, one row per intersection
target_71 = round(ORDER['phase1'] * CDOT['signals'])

pri = signals['priority']
summary = {
    'signals': N, 'aps': A, 'seenOnly': N - A, 'apsShare': A / N,
    'cdot': CDOT, 'listedNow': listed_now, 'listedAsOf': site_as_of, 'newSinceReport': site_new,
    'stations': int(len(city_st)), 'stationsAps': int((city_st['class'] == 'aps').sum()),
    'stationsSeenOnly': int((city_st['class'] == 'seen').sum()), 'stationsNoSignal': int((city_st['class'] == 'none').sum()),
    'accessible': int(len(acc)), 'accessibleSeenOnly': int((acc['class'] != 'aps').sum()),
    'nearStations': int(signals['near_cta'].sum()), 'nearStationsAps': int((signals['near_cta'] & signals['aps']).sum()),
    'priority': int(pri.sum()), 'priorityAps': int((pri & signals['aps']).sum()),
    'priorityByKind': {k: int(signals[f'near_{k}'].sum()) for k in KINDS},
    'otherAps': int((~pri & signals['aps']).sum()), 'other': int((~pri).sum()),
    'vision': round(vis_total), 'visionFar': round(vis_far), 'visionFarShare': vis_far / vis_total,
    'visionMedianM': round(vis_median_m),
    'byYear': {int(y): int(n) for y, n in by_year.items()},
    'planYear1': plan_year_1, 'target71': target_71,
    'yearsAtYear1Pace': (target_71 - listed_now) / CDOT['year1'],
    'monitor': MONITOR, 'order': ORDER,
}
pd.Series({k: v for k, v in summary.items() if not isinstance(v, dict)})

In [ ]:
# The finding printed on the plate, generated from the data so it can never drift from the numbers.
def walk_words(m: float) -> str:
    """'about a mile', 'about half a mile', else miles to one decimal."""
    mi = m / 1609.344
    for v, w in ((0.5, 'half a mile'), (1, 'a mile'), (1.5, 'a mile and a half'), (2, 'two miles')):
        if abs(mi - v) / v < 0.05:
            return f'about {w}'
    return f'{mi:.1f} miles'


FINDING = (
    f'At {summary["seenOnly"]:,} of the {N:,} signalized crossings mapped here, the only way to know when to '
    f'walk is to see the signal. Half of the roughly {round(vis_total, -3):,.0f} Chicagoans with a vision '
    f'difficulty live {walk_words(vis_median_m)} or more on foot from a crossing that does not depend on sight.'
)
print(FINDING)

## 6. QA

- **Placement.** Every APS on the City site carries a community area; the placed point must fall in
  it (a corner on a boundary may fall in its neighbour, so near misses within 100 m pass).
- **Universe.** The map's count against CDOT's 2,722; the difference is stated in the ledger.
- **Dates.** Installs in Plan Year 1 (29 May 2025 – 28 May 2026) must equal CDOT's 78.
- **Hand placements and additions** are listed for a look on a street map.

In [ ]:
areas = gpd.read_file(sorted(RAW_DIR.glob('community_areas_*.geojson'))[-1]).to_crs(CRS)[['community', 'geometry']]
placed_site = site[site['sid'] >= 0].merge(signals[['sid', 'x', 'y']], on='sid', suffixes=('_raw', ''))
pts = gpd.GeoDataFrame(placed_site, geometry=gpd.points_from_xy(placed_site['x'], placed_site['y']), crs=CRS)
pts['area'] = gpd.sjoin(pts, areas, predicate='within', how='left').groupby(level=0)['community_right'].first()
named = areas.set_index(areas['community'].str.upper())['geometry']
pts['off_m'] = [0 if str(a).upper() == c.upper() else named.get(c.upper(), shapely.Point()).distance(p)
                for a, c, p in zip(pts['area'], pts['community'], pts.geometry)]
KNOWN = {  # listed area is wrong or the corner sits on a boundary street; checked on a street map
    '435 N Michigan Tribune Crossing': 'Tribune Tower is in the Near North Side; the site lists Near West Side',
    '4640 W Diversey': 'Diversey is the Hermosa / Belmont Cragin boundary here',
}
bad = pts[(pts['off_m'] > 100) & ~pts['location'].isin(KNOWN)]
print('Placed outside their listed community area by more than 100 m:\n', bad[['location', 'community', 'area', 'off_m']].to_string())

qa = {
    'every site community area matches (within 100 m)': bad.empty,
    'Plan Year 1 installs = 78': plan_year_1 == CDOT['year1'],
    'map universe within 10% of CDOT': abs(N / CDOT['signals'] - 1) < 0.10,
    'two listed locations unplaced (Brickhouse, Edgebrook)': int((report['sid'] < 0).sum()) == 2,
    'station counts as Plate I': counts == DAY01_COUNTS,
    'site total = report + new': listed_now == CDOT['aps'] + site_new,
}
for name, ok in qa.items():
    log.info('%-50s %s', name, 'ok' if ok else 'FAILED')
print('Hand placed:', REWRITE)
print('Area exceptions:', KNOWN)
print('Added (no signal in OSM):', pd.concat([report, site]).loc[lambda d: d['how'].str.contains('no signal'), 'location'].drop_duplicates().tolist())
print(f'Universe: map {N:,} vs CDOT {CDOT["signals"]:,} ({N / CDOT["signals"] - 1:+.1%})')
assert all(qa.values()), 'QA failed; see log above'

## 7. QA view

A plain check, not the final design: every signalized intersection (rings), APS (dots), the
transit priority (dark rings) and stations by class.

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(14, 9), gridspec_kw={'width_ratios': [1.1, 1]})
gpd.GeoSeries([city], crs=CRS).plot(ax=ax, color='0.93')
s = signals
ax.scatter(s.x[~s.aps & ~s.priority], s.y[~s.aps & ~s.priority], s=3, facecolor='none', edgecolor='0.6', linewidth=0.4)
ax.scatter(s.x[~s.aps & s.priority], s.y[~s.aps & s.priority], s=3, facecolor='none', edgecolor='0.2', linewidth=0.5)
ax.scatter(s.x[s.aps], s.y[s.aps], s=9, color='tab:red')
cls = city_st['class'].map({'aps': 'tab:red', 'seen': 'tab:blue', 'none': 'tab:orange'})
ax.scatter(ST_XY[stations['inCity'], 0], ST_XY[stations['inCity'], 1], s=14, marker='s', facecolor='none', edgecolor=cls)
ax.set_axis_off(); ax.set_title('Signals (rings; dark = transit priority), APS (red), stations by class')
ax2.bar(list(summary['byYear']), list(summary['byYear'].values()), color='0.3')
ax2.set_title('APS installed by year (CDOT list to 12 Jun 2026)')
plt.show()

## 8. Export

`plate.json` carries the ledger, the finding, the summary, every signalized intersection (position in
address units, APS or not, install year, transit priority), the stations with their one-block
counts, and line totals for the chart. Keys are camelCase because the website reads them directly.

In [ ]:
def addr(xy: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    ll = gpd.GeoSeries(gpd.points_from_xy(*xy.T), crs=CRS).to_crs('EPSG:4326')
    return to_address(ll.y.to_numpy(), ll.x.to_numpy())


def pretty(location) -> str:
    """'N PULASKI RD-N ELSTON AVE' -> 'Pulaski & Elston'; the site's names are already readable."""
    if not isinstance(location, str):
        return ''
    if ADDRESS.match(location):
        return location.title().replace(' N ', ' N. ').replace(' S ', ' S. ').replace(' E ', ' E. ').replace(' W ', ' W. ')
    parts = list({norm(p): p for p in re.split(r'\s+-\s+|-|\s*&\s*', location) if p.strip()})
    tidy = lambda k: ' '.join(w.lower() if re.fullmatch(r'\d+(ST|ND|RD|TH)', w) else w.title() for w in bare(k).split())
    return ' & '.join(tidy(k) for k in parts)


sx, sy = addr(SIG_XY)
signals['ax'], signals['ay'] = sx, sy
signals['label'] = [pretty(s) if isinstance(s, str) else pretty(r) for s, r in zip(signals['site_name'], signals['name'])]
sig_out = [{
    'x': round(float(r.ax)), 'y': round(float(r.ay)), 'aps': bool(r.aps),
    **({'name': r.label, 'year': int(r.installed.year) if pd.notna(r.installed) else None} if r.aps else {}),
    'priority': bool(r.priority), 'nearStation': bool(r.near_cta), 'added': bool(r.added),
} for r in signals.itertuples()]

stations['x'], stations['y'] = to_address(stations['lat'], stations['lon'])
st_class = city_st['class'].reindex(stations.index).fillna('outside')
st_out = [{
    'id': r.map_id, 'name': r.name, 'short': r.short, 'lines': r.lines, 'ada': bool(r.step_free),
    'x': round(float(r.x), 1), 'y': round(float(r.y), 1), 'inCity': bool(r.inCity), 'class': c,
    'signals': int(r.signals), 'apsSignals': int(r.apsSignals),
    'toAps': None if not np.isfinite(r.toAps) else round(float(r.toAps)),
} for r, c in zip(stations.itertuples(), st_class)]

by_line = []
for line in LINES:
    on = stations[stations['lines'].map(lambda ls: line in ls)]
    oc = on[on['inCity']]
    by_line.append({'line': line, 'total': int(len(on)), 'inCity': int(len(oc)),
                    'aps': int((st_class[oc.index] == 'aps').sum()),
                    'seen': int((st_class[oc.index] == 'seen').sum()),
                    'none': int((st_class[oc.index] == 'none').sum())})

ledger = Ledger(
    source='CDOT, Chicago APS Semi-Annual Report (June 2026) and ChicagoAPS.org; OpenStreetMap',
    dataset_id='APS list (CDOT); OSM traffic signals',
    url=SITE_URL,
    pulled=site_pulled,
    units=f'{N:,} signalized intersections with pedestrian signals; {A} with APS',
    method=[
        f'APS list: CDOT report Appendix A (165, with install dates, as of {CDOT["asOf"]}) and ChicagoAPS.org '
        f'(as of {site_as_of}); two locations could not be placed',
        f'Signalized intersections from OSM ({osm_date}): signal and crosswalk nodes within 80 m, with a '
        f'signalized crosswalk; {int(signals["added"].sum())} added where an APS is listed but OSM has no signal',
        'One block = the court order’s 1/8 mile (201 m), walked on the OSM walk network',
        'Transit priority as the order: CTA and Metra stations, Pace Pulse stops, bus route crossings, streets under the L',
        'Vision difficulty: ACS 2020–2024 5-year B18103 by tract, allocated to 2020 blocks by population',
    ],
    caveats=[
        f'The map counts {N:,} signalized intersections; CDOT counts {CDOT["signals"]:,}',
        'An installed APS is not a compliant one: the court monitor found none of 103 inspected fully compliant',
        'Residents are estimates; distances run from each block’s center',
    ],
)
payload = {
    'summary': summary,
    'signals': sig_out,
    'stations': st_out,
    'byLine': by_line,
    'osmDate': osm_date,
    'rail': track,
    'gtfsFeed': GTFS_ZIP.stem.rsplit('_', 1)[-1],
    'sources': {'order': ORDER_URL, 'report': REPORT_URL, 'monitor': MONITOR_URL, 'site': SITE_URL},
}
write_plate(OUT_PATH, day=DAY, ledger=ledger, finding=FINDING, payload=payload)
log.info('Wrote %s (%.0f KB)', OUT_PATH, OUT_PATH.stat().st_size / 1024)

## Notes for the plate

- Finding: printed in section 5; it is generated from the data.
- The inset and the chart's station order are chosen in the renderer from `stations` and
  `reference/cta-line-sequences.json`, as Plate I.
- Next: the website reads `output/plate.json`; rebuild the site and export the PNG.